# 03 — Fine-tune GraphCodeBERT (binary vulnerability classifier)

**Run this notebook on Google Colab with a GPU** (`Runtime` → `Change runtime type` →
`T4 GPU`). Fine-tuning ~132K functions on CPU would take many hours; on a free
Colab T4 it takes roughly 1–2 hours for 3 epochs.

It also runs locally (CPU) for quick code checks — it auto-detects the
environment and adjusts data paths accordingly.

## Before running on Colab
Upload these 4 files (created in Phase 3) to your Google Drive at
`MyDrive/vulndetect-cpp/data/processed/`:
- `train.parquet`
- `val.parquet`
- `test.parquet`
- `class_weights.json`

(Just drag-and-drop the `data/processed` folder into Google Drive —
create the `vulndetect-cpp` folder first so the path matches.)

In [ ]:
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
print("Running on Colab:", IS_COLAB)

In [ ]:
if IS_COLAB:
    # Pinned to the versions requirements.txt uses locally, so the checkpoint
    # this notebook saves loads back on your machine without a version skew.
    %pip install -q transformers==5.16.1 tokenizers==0.23.2 safetensors==0.8.0 accelerate scikit-learn pandas pyarrow tqdm


In [ ]:
from pathlib import Path

if IS_COLAB:
    from google.colab import drive

    # drive.mount() raises ValueError("mount failed") when /content/drive is
    # already mounted, or when a restarted runtime left a stale mount behind.
    # Check for a live mount first, and force-remount only if that check fails.
    DRIVE_ROOT = Path("/content/drive")
    if (DRIVE_ROOT / "MyDrive").is_dir():
        print("Drive already mounted.")
    else:
        try:
            drive.mount(str(DRIVE_ROOT))
        except ValueError:
            print("Mount failed; retrying with force_remount ...")
            drive.mount(str(DRIVE_ROOT), force_remount=True)

    PROJECT = DRIVE_ROOT / "MyDrive" / "vulndetect-cpp"
    DATA_DIR = PROJECT / "data" / "processed"
    MODEL_OUT_DIR = PROJECT / "models" / "graphcodebert_finetuned"
else:
    DATA_DIR = Path("../data/processed")
    MODEL_OUT_DIR = Path("../models/graphcodebert_finetuned")

# Fail here with a clear message rather than deep inside pd.read_parquet.
missing = [f for f in ("train.parquet", "val.parquet", "test.parquet", "class_weights.json")
           if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f"Missing in {DATA_DIR}: {missing}. Upload them from data/processed/.")

MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR      :", DATA_DIR)
print("MODEL_OUT_DIR :", MODEL_OUT_DIR)


## Hyperparameters

- `MAX_LENGTH = 512` — GraphCodeBERT's hard limit (it's a RoBERTa-based model
  with 512 position embeddings). From `01_eda.ipynb`/token-length analysis,
  ~18% of functions in our dataset are longer than this and get truncated —
  a known limitation, not a bug.
- `SUBSET_SIZE` — set to a small number (e.g. `200`) for a **quick smoke test**
  (~2–3 min) to confirm the whole pipeline runs before committing to a full
  multi-hour run. Set to `None` for the real training run.
- `BATCH_SIZE = 16` fits comfortably on a T4 GPU (16GB) at `MAX_LENGTH=512`.
  Lower it (e.g. to 8) if you hit a CUDA out-of-memory error.

In [ ]:
MODEL_NAME = "microsoft/graphcodebert-base"

# GraphCodeBERT is RoBERTa-based and caps at 512 position embeddings, so this
# cannot simply be raised. The next cell measures what that costs us: long
# functions are truncated, and vulnerable functions are disproportionately long.
MAX_LENGTH = 512

# BATCH_SIZE is what fits in VRAM; GRAD_ACCUM_STEPS restores the effective
# batch size. Keep BATCH_SIZE * GRAD_ACCUM_STEPS == 16.
#   Colab T4 (16 GB)          -> 16 x 1
#   4 GB laptop GPU (3050 Ti) ->  4 x 4
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 1

NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
SUBSET_SIZE = None  # e.g. 200 for a quick smoke test; None = full dataset
SEED = 42

## 1. Load data

In [ ]:
import json

import numpy as np
import pandas as pd

train_df = pd.read_parquet(DATA_DIR / "train.parquet")
val_df = pd.read_parquet(DATA_DIR / "val.parquet")

with open(DATA_DIR / "class_weights.json") as f:
    class_weights = json.load(f)

if SUBSET_SIZE is not None:
    train_df = train_df.sample(min(SUBSET_SIZE, len(train_df)), random_state=42).reset_index(drop=True)
    val_df = val_df.sample(min(SUBSET_SIZE // 4, len(val_df)), random_state=42).reset_index(drop=True)

print(f"train: {len(train_df)}  val: {len(val_df)}")
print("class_weights:", class_weights)

### How much do we lose to the 512-token cap?

Worth knowing before training, because the answer is not "a rounding error"
and it is **not evenly distributed across the two classes**.

In [ ]:
from transformers import AutoTokenizer as _AutoTok

_tok = _AutoTok.from_pretrained(MODEL_NAME)
_sample = train_df.sample(min(2000, len(train_df)), random_state=SEED)
_lens = np.array([len(_tok(str(c), truncation=False)["input_ids"]) for c in _sample["func_before"]])

print(f"token length: median {np.median(_lens):.0f}  p90 {np.percentile(_lens, 90):.0f}  "
      f"p99 {np.percentile(_lens, 99):.0f}  max {_lens.max():,}")
print(f"truncated at {MAX_LENGTH}: {(_lens > MAX_LENGTH).mean():.1%} overall")
for _label in (0, 1):
    _m = _sample["vul"].to_numpy() == _label
    print(f"   vul={_label}: {(_lens[_m] > MAX_LENGTH).mean():.1%} truncated")

Vulnerable functions are truncated at roughly **twice** the rate of safe ones.
That is a systematic bias, not noise: the model often never sees the part of a
long function where the bug actually lives, and the truncation rate itself
correlates with the label. Treat it as a known ceiling on recall, and note it
when reporting results.

## 2. Tokenizer + PyTorch Dataset

Each `func_before` string is tokenized into `input_ids` (numeric token IDs)
and `attention_mask` (marks real tokens vs. padding). We pad every sequence
to `MAX_LENGTH` for simplicity — slightly less compute-efficient than dynamic
padding, but much easier to reason about as a beginner.

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class CodeDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.texts = df["func_before"].tolist()
        self.labels = df["vul"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


train_dataset = CodeDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = CodeDataset(val_df, tokenizer, MAX_LENGTH)
print(f"train_dataset: {len(train_dataset)}  val_dataset: {len(val_dataset)}")

## 3. DataLoaders — weighted sampling for train

`WeightedRandomSampler` uses the per-class weights from Phase 3
(`class_weights.json`) so each training batch sees roughly balanced classes,
instead of being ~94% "not vulnerable" by default.

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler

sample_weights = train_df["vul"].map(lambda v: class_weights[str(v)]).to_numpy()
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

pin = torch.cuda.is_available()
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, pin_memory=pin)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin)
print(f"train batches: {len(train_loader)}  val batches: {len(val_loader)}")

## 4. Model, optimizer, scheduler

In [ ]:
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)

# Mixed precision roughly halves activation memory and doubles throughput on a
# T4. It is a no-op on CPU, where this notebook is not practical anyway:
# measured at ~49 s/step on an 8-thread laptop CPU, i.e. ~330 h for 3 epochs.
USE_AMP = device.type == "cuda"
AMP_DTYPE = torch.float16
scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)

print("device:", device, "| AMP:", USE_AMP)
print(f"optimizer steps: {total_steps:,} "
      f"(batch {BATCH_SIZE} x grad-accum {GRAD_ACCUM_STEPS} = effective {BATCH_SIZE * GRAD_ACCUM_STEPS})")

## 5. Evaluation function

**Why not just accuracy?** With ~17:1 imbalance, a model that always predicts
"not vulnerable" scores ~94% accuracy while being useless. We track
precision/recall/F1 **for the vulnerable class specifically**, plus ROC-AUC.

In [ ]:
import numpy as np
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score


@torch.no_grad()
def predict_probs(loader):
    """Return (labels, P(vulnerable)) for a loader, as numpy arrays."""
    model.eval()
    all_labels, all_probs = [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            outputs = model(**batch)
        probs = F.softmax(outputs.logits.float(), dim=-1)[:, 1]
        all_labels.extend(batch["labels"].cpu().tolist())
        all_probs.extend(probs.cpu().tolist())
    return np.asarray(all_labels), np.asarray(all_probs)


def metrics_at(labels, probs, threshold=0.5):
    """Score a set of probabilities at one decision threshold.

    ROC-AUC is threshold-independent, so it is the same for every threshold;
    it is included to make each row self-contained.
    """
    preds = (probs >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", pos_label=1, zero_division=0
    )
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float("nan")  # only one class present (tiny smoke-test subsets)
    return {
        "threshold": round(float(threshold), 3),
        "accuracy": accuracy_score(labels, preds),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": auc,
    }


def evaluate(loader, threshold=0.5):
    """Convenience wrapper used by the training loop."""
    labels, probs = predict_probs(loader)
    return metrics_at(labels, probs, threshold)

## 6. Training loop

**Built to survive a disconnect.** Colab free drops sessions on idle (~90 min)
and on GPU quota, and this run takes hours. After every epoch the full training
state — weights, optimizer moments, scheduler, AMP scaler, epoch counter and
history — is written to Drive.

If the session dies, reconnect, run the notebook from the top, and this cell
resumes at the epoch it left off. Worst case you lose one epoch (~1 hour), not
the whole run.

The state file is ~1.5 GB and is overwritten each epoch, so it costs a couple
of minutes per epoch and 1.5 GB of Drive. Delete
`MyDrive/vulndetect-cpp/models/training_state.pt` to force a fresh start.

In [ ]:
import time

from tqdm.auto import tqdm

# Colab free disconnects on idle (~90 min) and on GPU quota, and this run takes
# hours. Full training state goes to Drive after every epoch so a disconnect
# costs at most one epoch instead of the whole run. Re-running this cell after
# reconnecting picks up where it stopped.
STATE_PATH = MODEL_OUT_DIR.parent / "training_state.pt"

start_epoch, best_f1, history = 0, -1.0, []

if STATE_PATH.exists():
    state = torch.load(STATE_PATH, map_location=device, weights_only=False)
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    scaler.load_state_dict(state["scaler"])
    start_epoch, best_f1, history = state["epoch"], state["best_f1"], state["history"]
    print(f"Resumed from {STATE_PATH} -> starting at epoch {start_epoch + 1}, best val F1 so far {best_f1:.4f}")
else:
    print(f"No checkpoint at {STATE_PATH}; starting from scratch.")

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    started = time.perf_counter()
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    for step, batch in enumerate(pbar, start=1):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            outputs = model(**batch)
            # Scale so accumulated gradients average rather than sum.
            loss = outputs.loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        running_loss += outputs.loss.item()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            # Gradients must be unscaled before clipping, or the threshold
            # would apply to the scaled values.
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        pbar.set_postfix(loss=running_loss / step)

    train_minutes = (time.perf_counter() - started) / 60
    metrics = evaluate(val_loader)
    metrics["epoch"] = epoch + 1
    metrics["train_loss"] = running_loss / len(train_loader)
    metrics["train_minutes"] = round(train_minutes, 1)
    history.append(metrics)
    print(f"Epoch {epoch + 1} ({train_minutes:.1f} min): {metrics}")

    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        model.save_pretrained(MODEL_OUT_DIR)
        tokenizer.save_pretrained(MODEL_OUT_DIR)
        print(f"  -> New best val F1={best_f1:.4f}, saved to {MODEL_OUT_DIR}")

    # ~1.5 GB (weights + AdamW moments). Written last so a crash mid-save
    # cannot corrupt the best-checkpoint directory above.
    torch.save(
        {
            "epoch": epoch + 1,
            "best_f1": best_f1,
            "history": history,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
        },
        STATE_PATH,
    )
    print(f"  -> training state saved to {STATE_PATH}")

print("\nTraining history:")
pd.DataFrame(history)

## 7. Final evaluation on the **test** set

The earlier version of this notebook stopped at validation. That is not enough:
validation drove checkpoint selection, so reporting it as the result is
reporting the number we optimised for.

The test split is touched exactly once, here, after training is finished.
Because `02_preprocessing.ipynb` split on `commit_id`, no commit in this set
appeared in training — these numbers are the honest ones.

In [ ]:
# Reload the best checkpoint (the training loop saved it on best val F1).
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(MODEL_OUT_DIR, num_labels=2).to(device)
model.eval()

test_df = pd.read_parquet(DATA_DIR / "test.parquet")
if SUBSET_SIZE is not None:
    test_df = test_df.sample(min(SUBSET_SIZE // 4, len(test_df)), random_state=SEED).reset_index(drop=True)

test_dataset = CodeDataset(test_df, tokenizer, MAX_LENGTH)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"test: {len(test_df):,} functions, {int(test_df['vul'].sum()):,} vulnerable "
      f"({test_df['vul'].mean():.2%})")

In [ ]:
val_labels, val_probs = predict_probs(val_loader)
test_labels, test_probs = predict_probs(test_loader)
print(f"collected probabilities: val {len(val_probs):,}, test {len(test_probs):,}")

### Tuning the decision threshold

`argmax` is a threshold of 0.5, which is rarely the F1-optimal cut on
imbalanced data. The threshold is chosen on **validation only** and then
applied unchanged to test — choosing it on test would be fitting to the set we
are reporting.

In [ ]:
# Sweep validation, pick the best-F1 threshold, then apply it to test as-is.
grid = np.arange(0.05, 0.96, 0.01)
val_f1s = [metrics_at(val_labels, val_probs, t)["f1"] for t in grid]
best_threshold = float(grid[int(np.argmax(val_f1s))])

print(f"best threshold on val: {best_threshold:.2f}  (F1 {max(val_f1s):.4f}, "
      f"vs {metrics_at(val_labels, val_probs, 0.5)['f1']:.4f} at 0.50)")

rows = []
for split, labels, probs in (("val", val_labels, val_probs), ("test", test_labels, test_probs)):
    for t in (0.5, best_threshold):
        rows.append({"split": split, **metrics_at(labels, probs, t)})

results = pd.DataFrame(rows)
print()
print(results.to_string(index=False))

The `test` row at the tuned threshold is the number to report. If tuning gains
a lot on validation but nothing on test, the gain was noise — say so rather
than quoting the validation figure.

### Reading these numbers

Report **precision / recall / F1 for the vulnerable class**, not accuracy. Only
~5% of functions are vulnerable, so a model that always answers "not
vulnerable" is already ~95% accurate and completely useless.

For context, published Big-Vul numbers vary enormously with the split policy:
papers using the standard random split report F1 up to ~0.9, while
cross-project evaluations land nearer 0.3–0.6. This split is commit-disjoint
but not project-disjoint (94.8% of test functions come from projects also in
train), so expect something between those poles. A number close to 0.95 would
mean leakage has crept back in.

In [ ]:
# Persist everything the README needs, so the numbers are never retyped by hand.
import math


def _clean(d):
    """NaN is valid Python but not valid JSON, so null it out."""
    return {k: (None if isinstance(v, float) and math.isnan(v) else v) for k, v in d.items()}


metrics_path = MODEL_OUT_DIR / "metrics.json"
payload = {
    "model": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "amp": USE_AMP,
    "learning_rate": LEARNING_RATE,
    "seed": SEED,
    "train_size": len(train_df),
    "best_threshold": best_threshold,
    "val": _clean(metrics_at(val_labels, val_probs, 0.5)),
    "val_tuned": _clean(metrics_at(val_labels, val_probs, best_threshold)),
    "test": _clean(metrics_at(test_labels, test_probs, 0.5)),
    "test_tuned": _clean(metrics_at(test_labels, test_probs, best_threshold)),
    "history": [_clean(h) for h in history],
}
metrics_path.write_text(json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8")
print("Saved:", metrics_path)

## 8. Getting the model back to your local machine

If `MODEL_OUT_DIR` was on Google Drive (`IS_COLAB=True`), the saved model is
already in your Drive at `MyDrive/vulndetect-cpp/models/graphcodebert_finetuned/`
— just download that folder from the Drive web UI (right-click → Download)
into your local `models/graphcodebert_finetuned/` so Phase 6 (ONNX export)
can find it.